# Notebook 8: Polynomial Regression

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours

**Theme:** Predicting house sale prices

---

## Why Does This Matter?

Linear regression assumes that the relationship between features and the target is a straight line. But real relationships are almost never perfectly linear:

- House price vs. age: new homes command a premium, middle-aged homes lose value, then vintage homes recover (nonlinear U-shape)
- Price vs. neighborhood income: nonlinear saturation effects at very high incomes
- Price vs. lot size: diminishing returns — going from 500 to 1,000 sq ft adds more value than going from 5,000 to 5,500 sq ft

Forcing a straight line through curved data leads to **systematic errors** — the model will be consistently wrong in predictable ways. Polynomial regression fixes this by letting the model learn curves.

---

## The Real-World Analogy: Fitting a Trend Line on a Sales Chart

Imagine you are a real estate analyst plotting house prices against median neighborhood income. You draw a straight trend line, but it clearly misses the curve — low-income areas have prices bunched near the bottom, while high-income areas have prices that accelerate upward.

Instead of forcing a ruler onto the chart, you bend the line: you let it curve to follow the data. That is polynomial regression. The key insight: **you are still using linear regression — you just added engineered features (x², x³, etc.) that let the line bend**.

Think of it like giving the model more "joints" to articulate. A degree-1 model is a rigid rod. A degree-3 model is a three-jointed arm — it can curve and adapt to more complex shapes. A degree-15 model has so many joints it can contort to pass through every single data point — but it will wildly mispredict any new point in between.

## Section 1: Polynomial Features — Plain English and Formula

### What Are Polynomial Features?

For a single feature x and polynomial degree d, we create the feature vector:

$$\phi(x) = [1, \; x, \; x^2, \; x^3, \; \ldots, \; x^d]$$

The model then fits:

$$\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3 + \cdots + \theta_d x^d$$

**This is still linear regression!** The model is linear in the parameters θ — we just transformed the input features. The OLS solution (XᵀX)⁻¹Xᵀy still applies with no modification.

### For Multiple Features

For 2 features (x₁, x₂) and degree 2, the feature vector becomes:

$$\phi(x_1, x_2) = [1, \; x_1, \; x_2, \; x_1^2, \; x_2^2, \; x_1 x_2]$$

The number of features grows rapidly with degree and number of input features:

$$\text{# features} = \binom{p + d}{d}$$

| Input features p | Degree d | Output features |
|---|---|---|
| 1 | 5 | 6 |
| 10 | 2 | 66 |
| 20 | 3 | 1,771 |
| 20 | 5 | 53,130 |

This feature explosion makes high-degree polynomials with many input features computationally expensive and prone to overfitting.

In [ ]:
# ── Imports and global style settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import comb                        # C(n,k) combination formula
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# Consistent visual theme
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.size': 9
})

np.random.seed(42)

# Demonstrate the feature count formula C(p+d, d)
print("Number of polynomial features (including bias):")
print(f"  p=1, d=1 (linear):  {comb(1+1, 1)+1} features")  # add 1 for bias
for p, d in [(1, 3), (1, 5), (10, 2), (20, 3), (20, 5)]:
    n_feat = comb(p + d, d)   # sklearn includes bias term in this count
    print(f"  p={p:2d} features, degree {d}: {n_feat:6,} polynomial features")

## Section 2: Generate the Demonstration Dataset

We use a noisy sine wave — the classic toy dataset for demonstrating underfitting and overfitting.

$$y = \sin(2\pi x) + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 0.2^2)$$

With only 20 training points, low-degree polynomials will underfit (miss the wave shape) and high-degree polynomials will overfit (wiggle through every noisy point). This gives us a clear, visual demonstration of the bias-variance trade-off.

In [ ]:
# ── Generate noisy sine wave data ──────────────────────────────────────────
n_train = 20       # deliberately small — makes overfitting obvious
n_test = 500       # dense test grid to show true function recovery
noise_std = 0.25   # realistic noise level

# Training points: random x in [0, 1]
X_train = np.sort(np.random.uniform(0, 1, n_train))
y_train = np.sin(2 * np.pi * X_train) + np.random.normal(0, noise_std, n_train)

# Validation set: another random draw of same size
X_val = np.sort(np.random.uniform(0, 1, n_train))
y_val = np.sin(2 * np.pi * X_val) + np.random.normal(0, noise_std, n_train)

# Dense test grid: smooth line for plotting the true function and predictions
X_test_grid = np.linspace(0, 1, n_test)
y_true_grid = np.sin(2 * np.pi * X_test_grid)  # noiseless ground truth

# Quick data overview
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_train, y_train, color='steelblue', zorder=3, s=50,
           label=f'Training data (n={n_train})')
ax.plot(X_test_grid, y_true_grid, 'k--', linewidth=1.5, alpha=0.7,
        label='True function: sin(2πx)')
ax.scatter(X_val, y_val, color='orange', marker='^', zorder=3, s=50,
           label=f'Validation data (n={n_train})')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Dataset: Noisy Sine Wave')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Training set: {n_train} points. Validation set: {n_train} points.")
print("Goal: find the polynomial degree that best recovers the sine wave.")

## Section 3: The Underfitting–Overfitting Spectrum

We now fit polynomials of degree 1 through 15 and observe how the fitted curve changes.

Key terms:
- **Underfitting (high bias):** the model is too simple to capture the true pattern. Error is high on both training and validation data.
- **Overfitting (high variance):** the model is too complex and memorizes noise in the training set. Error is low on training but high on validation data.
- **Sweet spot:** the degree where validation error is minimized.

In [ ]:
# ── Fit polynomials of degrees 1–15, record MSE ───────────────────────────
degrees = list(range(1, 16))
train_mses, val_mses = [], []
fitted_models = {}   # degree -> fitted LinearRegression model
fitted_polys = {}    # degree -> PolynomialFeatures transformer

X_train_2d = X_train.reshape(-1, 1)  # sklearn expects 2D input
X_val_2d = X_val.reshape(-1, 1)

for d in degrees:
    # Step 1: expand features (adds x, x², ..., xᵈ columns)
    poly = PolynomialFeatures(degree=d, include_bias=True)
    X_tr_poly = poly.fit_transform(X_train_2d)  # shape: (n, d+1)
    X_va_poly = poly.transform(X_val_2d)

    # Step 2: fit ordinary linear regression on the expanded features
    reg = LinearRegression(fit_intercept=False)  # bias already in poly features
    reg.fit(X_tr_poly, y_train)

    # Step 3: compute MSE on training and validation sets
    train_pred = reg.predict(X_tr_poly)
    val_pred = reg.predict(X_va_poly)
    train_mses.append(mean_squared_error(y_train, train_pred))
    val_mses.append(mean_squared_error(y_val, val_pred))

    fitted_models[d] = reg
    fitted_polys[d] = poly

best_degree = degrees[np.argmin(val_mses)]
print(f"Best degree by validation MSE: {best_degree}")
print(f"Training MSE at best degree:   {train_mses[best_degree-1]:.4f}")
print(f"Validation MSE at best degree: {val_mses[best_degree-1]:.4f}")
print(f"\nDegree  Train MSE   Val MSE")
for d, tr, va in zip(degrees, train_mses, val_mses):
    flag = " <-- best" if d == best_degree else ""
    print(f"  {d:2d}     {tr:8.4f}   {va:8.4f}{flag}")

In [ ]:
# ── Main visualization: 2×3 subplot grid for degrees 1,2,3,5,10,15 ─────────
show_degrees = [1, 2, 3, 5, 10, 15]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

X_grid_2d = X_test_grid.reshape(-1, 1)

labels = {
    1:  ('Degree 1 — Underfit',   'HIGH BIAS\nMisses the curve entirely'),
    2:  ('Degree 2 — Still poor',  'Slight improvement, still biased'),
    3:  ('Degree 3 — Good fit',    'Captures the wave shape well'),
    5:  ('Degree 5 — Best region', 'Near-optimal: low bias, low variance'),
    10: ('Degree 10 — Overfit',    'HIGH VARIANCE\nChases noise, wiggles wildly'),
    15: ('Degree 15 — Severe overfit', 'Extreme oscillation at boundaries')
}

for ax, d in zip(axes, show_degrees):
    title, annotation = labels[d]

    # Training data
    ax.scatter(X_train, y_train, color='steelblue', s=25, zorder=4, alpha=0.8)

    # True function
    ax.plot(X_test_grid, y_true_grid, 'k--', linewidth=1.2, alpha=0.5,
            label='True sin(2πx)')

    # Polynomial prediction
    poly_grid = fitted_polys[d].transform(X_grid_2d)
    y_pred_grid = fitted_models[d].predict(poly_grid)
    # Color: red for extreme cases, green for good cases
    color = 'crimson' if d in [1, 10, 15] else ('forestgreen' if d in [3, 5] else 'darkorange')
    ax.plot(X_test_grid, y_pred_grid, color=color, linewidth=2.0, label=f'Degree {d}')

    # Clip y-axis to avoid extreme oscillation distorting the plot
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')

    # Add MSE annotation in top-left corner
    tr_mse = train_mses[d - 1]
    va_mse = val_mses[d - 1]
    ax.text(0.02, 0.97, f'Train MSE: {tr_mse:.3f}\nVal MSE:   {va_mse:.3f}',
            transform=ax.transAxes, va='top', fontsize=7.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.text(0.5, 0.05, annotation, transform=ax.transAxes,
            ha='center', fontsize=7.5, color=color, alpha=0.9)

plt.suptitle('Polynomial Regression: Underfitting → Sweet Spot → Overfitting',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Section 4: Training vs Validation Error — The Bias-Variance Trade-off

The most important diagnostic plot in supervised learning. It shows:

- **Training error** (blue): always decreases as model complexity increases. A complex-enough model can memorize all training points.
- **Validation error** (orange): decreases at first (more flexibility helps), then increases (model starts memorizing noise). This U-shape identifies the sweet spot.
- **The gap** between training and validation error is the **overfitting gap** — it grows with model complexity.

**Bias-Variance decomposition:**
$$\text{Expected MSE} = \underbrace{\text{Bias}^2}_{\text{underfitting error}} + \underbrace{\text{Variance}}_{\text{overfitting error}} + \underbrace{\sigma^2}_{\text{irreducible noise}}$$

In [ ]:
# ── Training vs Validation MSE vs Degree ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: linear scale — shows the dramatic spike at high degrees
ax = axes[0]
ax.plot(degrees, train_mses, 'b-o', markersize=5, linewidth=2,
        label='Training MSE')
ax.plot(degrees, val_mses, 'r-s', markersize=5, linewidth=2,
        label='Validation MSE')
ax.axvline(best_degree, linestyle='--', color='green', linewidth=1.5,
           label=f'Best degree = {best_degree}')
# Shade the overfitting region (right of best degree)
ax.axvspan(best_degree, max(degrees), alpha=0.08, color='red',
           label='Overfitting zone')
ax.axvspan(min(degrees), best_degree, alpha=0.08, color='blue',
           label='Underfitting zone')
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('MSE')
ax.set_title('Train vs Validation MSE')
ax.set_ylim(bottom=0, top=min(max(val_mses)*1.05, 5.0))  # clip spike for readability
ax.legend(fontsize=8)
ax.set_xticks(degrees)

# Right: log scale — better reveals the gap between train and val error
ax2 = axes[1]
ax2.plot(degrees, train_mses, 'b-o', markersize=5, linewidth=2,
         label='Training MSE')
ax2.plot(degrees, val_mses, 'r-s', markersize=5, linewidth=2,
         label='Validation MSE')
ax2.axvline(best_degree, linestyle='--', color='green', linewidth=1.5,
            label=f'Best degree = {best_degree}')
ax2.set_xlabel('Polynomial Degree')
ax2.set_ylabel('MSE (log scale)')
ax2.set_title('Train vs Validation MSE (Log Scale)')
ax2.set_yscale('log')
ax2.legend(fontsize=8)
ax2.set_xticks(degrees)

plt.suptitle('Bias-Variance Trade-off: Find the Sweet Spot', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Degree 1 (underfit):  Train={train_mses[0]:.3f}, Val={val_mses[0]:.3f}")
print(f"Degree {best_degree} (sweet spot): Train={train_mses[best_degree-1]:.3f}, Val={val_mses[best_degree-1]:.3f}")
print(f"Degree 10 (overfit):  Train={train_mses[9]:.3f}, Val={val_mses[9]:.3f}")
print(f"Degree 15 (severe):   Train={train_mses[14]:.4f}, Val={val_mses[14]:.3f}")

## Section 5: Coefficient Magnitudes — An Overfitting Diagnostic

A subtle but important sign of overfitting: **the model coefficients become very large**.

Why? When the model overfits, it uses large positive and negative coefficients that nearly cancel each other out — this produces extreme oscillations that pass through noisy training points. The coefficients for degree-10 and degree-15 models are orders of magnitude larger than for degree-3 or degree-5 models.

This is why **regularization** works: by penalizing large coefficients, we force the model to find smoother solutions.

In [ ]:
# ── Coefficient magnitudes for degree-3, 5, 10 ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, d in zip(axes, [3, 5, 10]):
    coefs = fitted_models[d].coef_    # shape: (d+1,) including bias
    feature_names = [f'x^{i}' if i > 0 else 'bias' for i in range(d + 1)]
    colors = ['crimson' if abs(c) > 100 else 'steelblue' for c in coefs]
    ax.bar(feature_names, np.abs(coefs), color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Degree {d} — |Coefficients|', fontsize=11)
    ax.set_xlabel('Feature')
    ax.set_ylabel('|θ|')
    ax.set_yscale('log' if max(np.abs(coefs)) > 100 else 'linear')
    # Annotate max coefficient
    max_coef = np.max(np.abs(coefs))
    ax.text(0.98, 0.95, f'Max |θ|: {max_coef:.1f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            color='crimson' if max_coef > 100 else 'steelblue',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    # Rotate x labels for readability
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)

plt.suptitle('Coefficient Magnitudes: Overfitting = Huge Coefficients', fontsize=13)
plt.tight_layout()
plt.show()

print("Degree 3: coefficients are modest — model is smooth.")
print("Degree 10: coefficients are enormous — model oscillates wildly.")
print("Large coefficients are the fingerprint of overfitting.")

## Section 6: Regularization — Ridge Regression to the Rescue

**Problem:** A degree-10 polynomial overfits because the coefficients grow unconstrained.

**Solution:** Add a penalty term to the loss function that discourages large coefficients.

### Ridge Regression (L2 Regularization)

$$L_{\text{Ridge}}(\theta) = \underbrace{\frac{1}{n}\|X\theta - y\|^2}_{\text{fit to data}} + \underbrace{\lambda \|\theta\|^2}_{\text{L2 penalty}}$$

The λ parameter controls the strength of regularization:
- λ = 0: plain linear regression (no constraint)
- λ → ∞: all coefficients shrink to zero (model predicts the mean)
- λ optimal: smooth curve that generalizes well

### Lasso Regression (L1 Regularization)

$$L_{\text{Lasso}}(\theta) = \frac{1}{n}\|X\theta - y\|^2 + \lambda \|\theta\|_1$$

Lasso promotes **sparsity**: many coefficients become exactly zero. This performs automatic feature selection — useful when you have many irrelevant polynomial terms.

| Property | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Penalty | λ||θ||² | λ||θ||₁ |
| Coefficients | Shrunk but nonzero | Many become exactly 0 |
| Feature selection | No | Yes |
| Best when | All features contribute | Many irrelevant features |

In [ ]:
# ── Ridge vs No Regularization on Degree-10 Polynomial ────────────────────
d_overfit = 10
poly10 = PolynomialFeatures(degree=d_overfit, include_bias=True)
X_tr_p10 = poly10.fit_transform(X_train_2d)
X_grid_p10 = poly10.transform(X_test_grid.reshape(-1, 1))

# No regularization (plain OLS on degree-10 features)
ols10 = LinearRegression(fit_intercept=False).fit(X_tr_p10, y_train)

# Ridge with several lambda values to show the smoothing effect
ridge_lambdas = [0.0001, 0.01, 1.0, 100.0]
ridge_models = {}
for lam in ridge_lambdas:
    # sklearn Ridge uses 'alpha' for λ
    ridge = Ridge(alpha=lam, fit_intercept=False)
    ridge.fit(X_tr_p10, y_train)
    ridge_models[lam] = ridge

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: OLS vs selected Ridge models
ax = axes[0]
ax.scatter(X_train, y_train, color='steelblue', s=30, zorder=4, label='Training data')
ax.plot(X_test_grid, y_true_grid, 'k--', linewidth=1.2, alpha=0.5, label='True sin(2πx)')
ax.plot(X_test_grid, np.clip(ols10.predict(X_grid_p10), -3, 3),
        'crimson', linewidth=2, label='OLS (no reg.) — overfit')
ax.plot(X_test_grid, ridge_models[0.01].predict(X_grid_p10),
        'darkorange', linewidth=2, linestyle='--', label='Ridge λ=0.01')
ax.plot(X_test_grid, ridge_models[1.0].predict(X_grid_p10),
        'forestgreen', linewidth=2, linestyle='-.', label='Ridge λ=1.0')
ax.plot(X_test_grid, ridge_models[100.0].predict(X_grid_p10),
        'purple', linewidth=2, linestyle=':', label='Ridge λ=100 (underfit)')
ax.set_ylim(-2.5, 2.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Degree-10 Polynomial: OLS vs Ridge Regularization')
ax.legend(fontsize=7.5)

# Right: how coefficient magnitudes shrink as lambda increases
ax2 = axes[1]
all_lambdas = [0] + ridge_lambdas + [1000]
coef_norms = []
for lam in all_lambdas:
    if lam == 0:
        coef_norms.append(np.linalg.norm(ols10.coef_))
    else:
        ridge_tmp = Ridge(alpha=lam, fit_intercept=False).fit(X_tr_p10, y_train)
        coef_norms.append(np.linalg.norm(ridge_tmp.coef_))

ax2.plot([str(l) for l in all_lambdas], coef_norms, 'o-', color='steelblue',
         linewidth=2, markersize=7)
ax2.set_xlabel('Ridge λ (regularization strength)')
ax2.set_ylabel('||θ||₂ (coefficient norm)')
ax2.set_title('Coefficient Norm Shrinks as λ Increases')
ax2.set_yscale('log')

plt.suptitle('Regularization: Ridge Smooths the Overfit', fontsize=13)
plt.tight_layout()
plt.show()

print("As λ increases: coefficients shrink → curve becomes smoother.")
print("Too large λ: curve becomes too flat (underfits). Must be tuned with CV.")

## Section 7: Cross-Validation — The Right Way to Select Degree

Using a single train/val split to pick the best degree risks:
1. The validation set may not be representative (small n = high variance in estimate)
2. The model is effectively "overfit" to the validation set through our manual selection

**K-Fold Cross-Validation** gives a more reliable estimate:
1. Split training data into K folds (typically K=5 or K=10)
2. For each fold: train on K-1 folds, validate on the held-out fold
3. Average the K validation errors
4. Report mean ± standard deviation of CV error

This uses all training data for both training and validation, and the ± std tells us how stable the estimate is.

In [ ]:
# ── 5-Fold Cross-Validation for Degree Selection ───────────────────────────
degrees_cv = list(range(1, 12))   # degrees 1–11 for CV
cv_means, cv_stds = [], []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for d in degrees_cv:
    # Build a pipeline: PolynomialFeatures → StandardScaler → LinearRegression
    # Scaling is important for numerical stability with high-degree polynomials
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=True)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    # cross_val_score returns negative MSE (sklearn convention for maximization)
    cv_scores = cross_val_score(
        pipe, X_train_2d, y_train,
        cv=kf, scoring='neg_mean_squared_error'
    )
    cv_mses = -cv_scores    # flip sign to get positive MSE
    cv_means.append(cv_mses.mean())
    cv_stds.append(cv_mses.std())

best_degree_cv = degrees_cv[np.argmin(cv_means)]

# Plot CV MSE with error bars (±1 std)
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(degrees_cv, cv_means, yerr=cv_stds, fmt='o-', capsize=5,
            color='steelblue', linewidth=2, markersize=6, elinewidth=1.5,
            label='5-fold CV MSE ± std')
ax.axvline(best_degree_cv, linestyle='--', color='forestgreen', linewidth=1.8,
           label=f'Best degree = {best_degree_cv}')
ax.fill_between(degrees_cv,
                [m - s for m, s in zip(cv_means, cv_stds)],
                [m + s for m, s in zip(cv_means, cv_stds)],
                alpha=0.15, color='steelblue')
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('CV MSE')
ax.set_title('5-Fold Cross-Validation MSE by Polynomial Degree')
ax.set_xticks(degrees_cv)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best degree by 5-fold CV: {best_degree_cv}")
print(f"CV MSE at best degree: {cv_means[best_degree_cv-1]:.4f} ± {cv_stds[best_degree_cv-1]:.4f}")
print("\nCV MSE by degree:")
for d, m, s in zip(degrees_cv, cv_means, cv_stds):
    flag = " <-- best" if d == best_degree_cv else ""
    print(f"  Degree {d:2d}: {m:.4f} ± {s:.4f}{flag}")

## Section 8: House Price Application — California Housing Dataset

Now we apply polynomial regression to real-world house price data: the California Housing dataset from scikit-learn.

We focus on the relationship between **median income** (in tens of thousands of dollars) and **median house value** (in hundreds of thousands of dollars). This relationship is known to be nonlinear — income explains house value well but with diminishing returns at the high end.

In [ ]:
# ── Load California Housing Dataset ───────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Use median_income as the single predictor for visualization clarity
# Cap house value at 5.0 (the dataset is capped at $500,001 — a known artifact)
df_clean = df[df['MedHouseVal'] < 5.0].copy()

# Sample 2,000 points for speed; the full 20k points gives same trends
df_sample = df_clean.sample(n=2000, random_state=42)

X_house = df_sample[['MedInc']].values          # shape (2000, 1)
y_house = df_sample['MedHouseVal'].values        # shape (2000,)

# Train/test split
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Plot the raw relationship
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_house[:, 0], y_house, alpha=0.15, color='steelblue',
           edgecolors='none', s=12, label='California housing data (sample)')
ax.set_xlabel('Median Income (tens of thousands USD)')
ax.set_ylabel('Median House Value (hundreds of thousands USD)')
ax.set_title('California Housing: Income vs House Value')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Training samples: {len(X_h_train)}, Test samples: {len(X_h_test)}")
print(f"Income range: ${X_house.min()*10:.0f}k to ${X_house.max()*10:.0f}k")
print(f"House value range: ${y_house.min()*100:.0f}k to ${y_house.max()*100:.0f}k")

In [ ]:
# ── Fit Polynomial Degrees 1–6 with 5-Fold CV on California Housing ────────
house_degrees = list(range(1, 7))
house_cv_means, house_cv_stds = [], []
house_test_mses = []

kf_house = KFold(n_splits=5, shuffle=True, random_state=42)

for d in house_degrees:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=d, include_bias=True)),
        ('scaler', StandardScaler()),          # necessary: income values vary widely
        ('reg',    LinearRegression())
    ])
    # Cross-validation on training set
    cv_scores = cross_val_score(
        pipe, X_h_train, y_h_train,
        cv=kf_house, scoring='neg_mean_squared_error'
    )
    house_cv_means.append(-cv_scores.mean())
    house_cv_stds.append(cv_scores.std())

    # Also compute test MSE for reference (we pretend not to look during selection)
    pipe.fit(X_h_train, y_h_train)
    y_pred_test = pipe.predict(X_h_test)
    house_test_mses.append(mean_squared_error(y_h_test, y_pred_test))

best_house_deg = house_degrees[np.argmin(house_cv_means)]
print(f"Best polynomial degree for California Housing: {best_house_deg}")
print(f"\n{'Degree':>8} {'CV MSE':>10} {'±Std':>10} {'Test MSE':>10}")
print("-" * 42)
for d, cm, cs, tm in zip(house_degrees, house_cv_means, house_cv_stds, house_test_mses):
    flag = " <--" if d == best_house_deg else ""
    print(f"{d:>8} {cm:>10.4f} {cs:>10.4f} {tm:>10.4f}{flag}")

In [ ]:
# ── Visualize polynomial fits on California Housing Data ──────────────────
# Sort income values for smooth curve plotting
x_income_grid = np.linspace(X_house.min(), X_house.max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: fitted curves for each degree
ax = axes[0]
ax.scatter(X_house[:, 0], y_house, alpha=0.1, color='lightgray',
           edgecolors='none', s=8, label='Data')

palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(house_degrees)))
for d, col in zip(house_degrees, palette):
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=d, include_bias=True)),
        ('scaler', StandardScaler()),
        ('reg',    LinearRegression())
    ])
    pipe.fit(X_h_train, y_h_train)
    y_pred_curve = pipe.predict(x_income_grid)
    lw = 2.5 if d == best_house_deg else 1.2
    ls = '-' if d == best_house_deg else '--'
    label = f'Degree {d}' + (' ← best' if d == best_house_deg else '')
    ax.plot(x_income_grid.ravel(), y_pred_curve, color=col,
            linewidth=lw, linestyle=ls, label=label)

ax.set_xlabel('Median Income (tens of thousands)')
ax.set_ylabel('Median House Value (hundreds of thousands)')
ax.set_title('Polynomial Fits: California Housing')
ax.set_ylim(y_house.min() - 0.2, y_house.max() + 0.2)
ax.legend(fontsize=7.5)

# Right: CV MSE vs degree
ax2 = axes[1]
ax2.errorbar(house_degrees, house_cv_means, yerr=house_cv_stds,
             fmt='o-', capsize=5, color='steelblue', linewidth=2,
             markersize=7, label='5-fold CV MSE ± std')
ax2.plot(house_degrees, house_test_mses, 's--', color='crimson',
         linewidth=1.5, markersize=6, label='Test MSE')
ax2.axvline(best_house_deg, linestyle='--', color='forestgreen', linewidth=1.5,
            label=f'Best degree = {best_house_deg}')
ax2.set_xlabel('Polynomial Degree')
ax2.set_ylabel('MSE')
ax2.set_title('CV MSE vs Degree (California Housing)')
ax2.set_xticks(house_degrees)
ax2.legend(fontsize=8)

plt.suptitle('Polynomial Regression on Real House Price Data', fontsize=13)
plt.tight_layout()
plt.show()

## Section 9: Summary — When to Use Polynomial Regression

In [ ]:
# ── Comparison summary table ───────────────────────────────────────────────
summary = pd.DataFrame({
    'Model': [
        'Linear (degree 1)',
        'Polynomial degree 3–5',
        'Polynomial degree 10+',
        'Polynomial + Ridge',
        'Polynomial + Lasso'
    ],
    'Bias': ['High', 'Low', 'Very low', 'Medium', 'Medium'],
    'Variance': ['Low', 'Medium', 'Very high', 'Low', 'Low'],
    'Generalization': ['Poor if curved', 'Good', 'Poor', 'Good', 'Good'],
    'Use When': [
        'True relationship is linear',
        'Mild nonlinearity, small n',
        'Never without regularization',
        'Nonlinear + many features',
        'Nonlinear + sparse features'
    ]
})
print(summary.to_string(index=False))

print("\n" + "="*65)
print("POLYNOMIAL REGRESSION KEY FORMULAS")
print("="*65)
print("Feature expansion:  φ(x) = [1, x, x², ..., xᵈ]")
print("Model:              ŷ = θ₀ + θ₁x + θ₂x² + ... + θᵈxᵈ")
print("Feature count:      C(p+d, d) for p inputs, degree d")
print("Ridge loss:         MSE + λ||θ||²")
print("Lasso loss:         MSE + λ||θ||₁")
print("\nSelection strategy:")
print("  1. Start with low degree (d=2 or 3)")
print("  2. Use K-fold CV to find optimal degree")
print("  3. If overfitting persists, add Ridge/Lasso")
print("  4. For complex nonlinearity, consider tree-based models")

print(f"\nResults on this notebook's datasets:")
print(f"  Sine wave:          best degree = {best_degree_cv} (by 5-fold CV)")
print(f"  California housing: best degree = {best_house_deg} (by 5-fold CV)")

## Self-Check Questions

Test your understanding. Try to answer each question before revealing the solution.

---

**Question 1:** A degree-10 polynomial achieves training MSE = 0.001 but validation MSE = 15.3. What do the coefficients likely look like? What would you do?

<details>
<summary>Reveal Answer</summary>

**Coefficients:** The model is severely overfitting. The coefficients are almost certainly very large — likely in the thousands or millions — with alternating signs that cancel each other out to produce extreme polynomial oscillations. The model has memorized the training noise rather than the underlying pattern.

**What to do:**
1. **Reduce degree:** Try degrees 3, 5, 7 and use cross-validation to find the best.
2. **Add regularization:** Apply Ridge regression (`Ridge(alpha=...)`) to the degree-10 model. Tune λ with cross-validation. Even with degree-10 features, Ridge can produce a smooth, well-generalizing curve.
3. **Increase training data:** More data makes it harder to overfit.
4. **Check the gap:** If training MSE = 0.001 but validation MSE = 15.3, the overfitting gap is enormous. This is a red flag that the model has essentially memorized the training labels.

</details>

---

**Question 2:** You add degree-5 polynomial features to a dataset with 20 original features. How many new features do you get? (Use the formula C(20+5, 5))

<details>
<summary>Reveal Answer</summary>

Using the formula for the number of polynomial features (including the bias term):

$$\binom{p + d}{d} = \binom{20 + 5}{5} = \binom{25}{5} = \frac{25!}{5! \cdot 20!} = 53,130$$

Starting from 20 original features, you end up with **53,130 polynomial features** (including all cross-products like x₁²x₃x₇, etc.). This is the feature explosion problem.

With only 53,130 features from 20 inputs, fitting an ordinary linear regression requires either a very large dataset or strong regularization. This is why high-degree polynomial features with many inputs are rarely used in practice — kernel methods (like SVMs with polynomial kernels) solve this problem more elegantly.

</details>

---

**Question 3:** Ridge regression with a high λ on a degree-10 polynomial — what happens to the fitted curve as λ → ∞?

<details>
<summary>Reveal Answer</summary>

As λ → ∞, the Ridge penalty term λ||θ||² dominates the loss function completely. The only way to minimize the total loss is to shrink all coefficients toward zero.

**As λ → ∞:**
- All polynomial coefficients θ₁, θ₂, ..., θᵈ → 0
- The only remaining term is the bias θ₀, which Ridge does not penalize (by convention)
- θ₀ → ȳ (the mean of the training labels)
- The fitted curve becomes a **horizontal flat line at y = ȳ**

This is maximum underfitting: the model ignores all features entirely and just predicts the training mean. The optimal λ is somewhere between 0 (overfit) and ∞ (underfit) — found by cross-validation.

</details>

---

**Question 4:** Why is polynomial regression still called "linear" regression even though it fits curves?

<details>
<summary>Reveal Answer</summary>

"Linear" in "linear regression" refers to **linearity in the parameters θ**, not linearity in the input features x.

The model is:
$$\hat{y} = \theta_0 \cdot 1 + \theta_1 \cdot x + \theta_2 \cdot x^2 + \theta_3 \cdot x^3$$

If we define new features φ₀ = 1, φ₁ = x, φ₂ = x², φ₃ = x³, then:
$$\hat{y} = \theta_0 φ_0 + \theta_1 φ_1 + \theta_2 φ_2 + \theta_3 φ_3 = \boldsymbol{\theta}^\top \boldsymbol{\phi}$$

This is a **linear** combination of the transformed features. Because the model is linear in θ, the OLS solution still applies: θ = (ΦᵀΦ)⁻¹Φᵀy, where Φ is the design matrix of polynomial features. All linear regression theory (convex loss surface, closed-form solution, Gauss-Markov theorem) carries over exactly.

The nonlinearity is in the **feature transformation** x → φ(x), not in the model itself.

</details>